# Quantum Circuit Dataset Generation with RL-NoiseModel

This notebook demonstrates how to use the refactored `rlnoise` package to generate datasets of quantum circuits with custom noise models.

**Features demonstrated:**
- Generating random and Clifford circuits
- Applying custom noise models
- Creating datasets for machine learning
- Saving and loading datasets
- Evaluation and randomized benchmarking datasets

## 1. Setup and Imports

In [1]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import qibo
qibo.set_backend("numpy")

# RL Noise imports
from rlnoise import (
    DatasetConfig,
    NoiseConfig,
    CircuitDataset,
    DatasetGenerator,
    CircuitGenerator,
    CircuitEncoder,
    QuantumNoiseModel,
)

# Import ExperimentConfig for later use
from rlnoise.config import ExperimentConfig

print("Setup complete!")

[Qibo 0.2.23|INFO|2026-02-26 17:28:06]: Using numpy backend on /CPU:0


Setup complete!


## 2. Basic Dataset Generation

Let's start with a simple example: generating a dataset of single-qubit quantum circuits with noise.

**Note:** `primitive_gates` is now specified in `DatasetConfig`, and `DatasetGenerator` automatically validates that the noise config's gate references (e.g., `depol_on_gate`) are compatible with the primitive gates.

In [ ]:
# Configure noise model (uniform noise across all qubits)
noise_config = NoiseConfig(
    depolarizing=0.02,              # Depolarizing noise strength (same for all qubits)
    damping=0.03,                   # Amplitude damping probability
    coherent_x=0.04,                # Coherent X error rate
    coherent_y=0.02,                # Coherent Y error rate
    x_coherent_on_gate=["rx"],      # Apply X coherent errors on RX gates
    z_coherent_on_gate=["rz"],      # Apply Z coherent errors on RZ gates
    damping_on_gate=["rx"],         # Apply damping errors on RX gates
    depol_on_gate=["rz"],           # Apply depolarizing errors on RZ gates
)

print(noise_config)


  NoiseConfig
  System:
    • Qubits:          1
    • Primitive Gates: [rx, rz]
  
  Noise Parameters:
    • Depolarizing:    0.0200
    • Damping:         0.0300
    • Coherent X:      0.0400
    • Coherent Y:      0.0200
  
  Noise Application:
    • X coherent → rx
    • Z coherent → rz
    • Damping    → rx
    • Depolarize → rz


In [ ]:
# Configure dataset parameters
dataset_config = DatasetConfig(
    n_circuits=50,              # Number of circuits to generate
    qubits=1,                   # Single-qubit circuits
    moments=10,                 # Circuit depth (number of gate layers)
    primitive_gates=["rx", "rz"],  # Available gate types
    clifford=True,              # Use Clifford gates (quantized angles)
    mixed=False,                # All circuits from same distribution
    eval_depth=15,              # Depth to use for evaluation
    eval_size=50,               # Number of circuits to use for evaluation
)

print(dataset_config)


  DatasetConfig
  Training Dataset:
    • Circuits:       50
    • Qubits:         1
    • Moments:        10
    • Type:           Clifford
  
  Evaluation Dataset:
    • Circuits:       50
    • Moments:        15


In [4]:
# Create generator and generate dataset
generator = DatasetGenerator(dataset_config, noise_config)
dataset = generator.generate(verbose=True)

print(dataset)

Generating 50 circuits...
Applying noise model...
Encoding circuits...
Dataset generated: 50 circuits, 1 qubits, 10 moments

  CircuitDataset
    • Total Circuits:      50
    • Qubits:              1
    • Moments (depth):     10
    • Encoding dimension:  8
    • Circuit shape:       (10, 1, 8)


## 3. Inspect Dataset Structure

The dataset contains:
- **Circuits**: Encoded representations (n_moments, n_qubits, encoding_dim)
- **Labels**: Density matrices from noisy circuit execution

In [ ]:
# Get a single sample
circuit_encoding, density_matrix = dataset[0]

# Generate a simple circuit for visualization using the dataset config
circuit_gen = CircuitGenerator(dataset_config)
example_circuit = circuit_gen.generate_random_circuit()

print("="*60)
print("  QIBO CIRCUIT (Original)")
print("="*60)
print(example_circuit.draw())

# Encode the circuit
encoder = CircuitEncoder(dataset_config.primitive_gates)
encoded_circuit = encoder.encode_circuit(example_circuit)

print("\n" + "="*60)
print("  CIRCUIT ENCODING (for ML)")
print("="*60)
print(f"Shape: {encoded_circuit.shape} (moments × qubits × encoding_dim)")
print(f"Data type: {encoded_circuit.dtype}")
print("\nFirst moment encoding:")
print(encoded_circuit[0])
print("\nEncoding indices:")
for key, idx in encoder.encoding_map.items():
    print(f"  {key:15s} → index {idx}")

# Apply noise and get density matrix
noise_model = QuantumNoiseModel(noise_config, dataset_config.qubits)
noisy_circuit = noise_model.apply(example_circuit)
result = noisy_circuit()

print("\n" + "="*60)
print("  DENSITY MATRIX (Label)")
print("="*60)
print(f"Shape: {result.shape}")
print(f"Trace: {np.trace(result):.6f} (should be ~1.0)")
print(f"Hermitian: {np.allclose(result, result.conj().T)}")
print(f"\nDensity matrix:")
print(result)

AttributeError: qubits

## 4. Save and Load Dataset

Datasets can be easily saved to disk and loaded later.

In [ ]:
# Create output directory
output_dir = Path("example_datasets")
output_dir.mkdir(exist_ok=True)

# Save dataset
save_path = output_dir / "basic_dataset"
dataset.save(str(save_path))
print(f"Dataset saved to {save_path}.npz")

# Load dataset
loaded_dataset = CircuitDataset.load(str(save_path) + ".npz")
print(f"Dataset loaded: {len(loaded_dataset)} circuits")

# Verify data integrity
assert len(loaded_dataset) == len(dataset)
print("✓ Data integrity verified")

## 5. Multi-Qubit Circuits

Generate datasets with multi-qubit circuits and two-qubit gates (CZ).

In [ ]:
# Configure 2-qubit dataset
multiqubit_config = DatasetConfig(
    n_circuits=30,
    qubits=2,
    moments=15,
    primitive_gates=["rx", "rz", "cz"],  # Include CZ gates (allowed with 2+ qubits)
    clifford=True,
)

# Update noise config to match the gates
multiqubit_noise = NoiseConfig(
    depolarizing=0.03,
    damping=0.02,
)

print("Multi-qubit dataset config:")
print(multiqubit_config)
print("\nMulti-qubit noise config:")
print(multiqubit_noise)

# Generate
generator_2q = DatasetGenerator(multiqubit_config, multiqubit_noise)
dataset_2q = generator_2q.generate(verbose=True)

print(f"\n2-qubit density matrix shape: {dataset_2q.labels[0].shape}")
print(f"Expected: (4, 4) for 2 qubits")

## 5b. Per-Qubit Noise Parameters

Noise parameters can be specified per qubit using lists. This allows asymmetric noise models.

In [ ]:
# Configure 3-qubit system with different noise per qubit
perqubit_config = DatasetConfig(
    n_circuits=20,
    qubits=3,
    moments=12,
    primitive_gates=["rx", "rz", "cz"],
    clifford=True,
)

# Per-qubit noise: qubit 0 is noisier than qubits 1 and 2
perqubit_noise = NoiseConfig(
    depolarizing=[0.05, 0.02, 0.02],  # Qubit 0 has more depolarizing noise
    damping=[0.04, 0.01, 0.01],       # Qubit 0 has more damping
    coherent_x=[0.06, 0.03, 0.03],    # Qubit 0 has more coherent X error
    coherent_y=[0.03, 0.01, 0.01],    # Qubit 0 has more coherent Y error
)

print("Per-qubit configuration:")
print(perqubit_config)
print("\nPer-qubit noise configuration:")
print(perqubit_noise)

# Generate dataset
generator_perqubit = DatasetGenerator(perqubit_config, perqubit_noise)
dataset_perqubit = generator_perqubit.generate(verbose=True)

print(f"\n3-qubit density matrix shape: {dataset_perqubit.labels[0].shape}")
print(f"Expected: (8, 8) for 3 qubits")

## 6. Evaluation Dataset

Create a separate evaluation dataset with different circuit depth.

In [ ]:
# Generate evaluation set with different depth
eval_dataset = generator.generate_evaluation_set(
    eval_depth=20,  # Deeper circuits than training
    eval_size=25,   # Fewer circuits
    verbose=True
)

print(f"\nEvaluation dataset: {len(eval_dataset)} circuits")
print(f"Training depth: {dataset.circuits[0].shape[0]} moments")
print(f"Evaluation depth: {eval_dataset.circuits[0].shape[0]} moments")

## 7. Randomized Benchmarking Dataset

Generate multiple datasets with increasing circuit depth for randomized benchmarking experiments.

In [ ]:
# Generate RB datasets
rb_datasets = generator.generate_rb_dataset(
    start=3,                    # Starting depth
    stop=15,                    # Ending depth (exclusive)
    step=3,                     # Step size
    n_circuits_per_depth=20,   # Circuits per depth level
    verbose=True
)

print(f"\nGenerated {len(rb_datasets)} RB datasets")
for i, ds in enumerate(rb_datasets):
    depth = 3 + i * 3
    print(f"  Depth {depth}: {len(ds)} circuits")

## 8. Train/Validation Split

Split a dataset into training and validation sets.

In [ ]:
# Split dataset
train_dataset, val_dataset = dataset.split(val_fraction=0.2)

print(f"Original dataset: {len(dataset)} circuits")
print(f"Training set: {len(train_dataset)} circuits")
print(f"Validation set: {len(val_dataset)} circuits")

## 9. Visualize Dataset Statistics

Analyze properties of the generated dataset.

In [ ]:
# Analyze density matrix properties
traces = [np.abs(np.trace(dm)) for dm in dataset.labels]
purities = [np.abs(np.trace(dm @ dm)) for dm in dataset.labels]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot traces
axes[0].hist(traces, bins=20, edgecolor="black", alpha=0.7)
axes[0].axvline(1.0, color="red", linestyle="--", label="Ideal (1.0)")
axes[0].set_xlabel("Trace")
axes[0].set_ylabel("Count")
axes[0].set_title("Density Matrix Traces")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot purities
axes[1].hist(purities, bins=20, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("Purity")
axes[1].set_ylabel("Count")
axes[1].set_title("Density Matrix Purities")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean trace: {np.mean(traces):.6f}")
print(f"Mean purity: {np.mean(purities):.6f}")
print(f"Purity range: [{np.min(purities):.4f}, {np.max(purities):.4f}]")

## 10. Advanced: Custom Configuration from JSON

You can also create configurations from JSON dictionaries, compatible with the old config files.

In [ ]:
# JSON-style configuration
config_dict = {
    "dataset": {
        "n_circuits": 100,
        "qubits": 1,
        "moments": 10,
        "primitive_gates": ["rx", "rz"],
        "clifford": True,
        "eval_size": 50,
        "eval_depth": 15,
    },
    "noise": {
        "depolarizing": 0.02,
        "damping": 0.03,
        "coherent_x": 0.04,
        "coherent_y": 0.02,
    }
}

# Create from dictionary
exp_config = ExperimentConfig.from_json(config_dict)
generator_from_json = DatasetGenerator.from_config(exp_config)

# Generate dataset
dataset_from_json = generator_from_json.generate(verbose=False)
print(f"Dataset from JSON config: {len(dataset_from_json)} circuits")

## 11. Non-Clifford (Arbitrary Angles) Circuits

Generate circuits with arbitrary rotation angles instead of quantized Clifford angles.

In [ ]:
# Non-Clifford configuration
non_clifford_config = DatasetConfig(
    n_circuits=30,
    qubits=1,
    moments=8,
    primitive_gates=["rx", "rz"],
    clifford=False,  # Arbitrary angles
)

generator_non_clifford = DatasetGenerator(non_clifford_config, noise_config)
dataset_non_clifford = generator_non_clifford.generate(verbose=True)

print(f"\nNon-Clifford dataset generated: {len(dataset_non_clifford)} circuits")

## Summary

This notebook demonstrated:

1. ✓ Basic dataset generation with custom noise
2. ✓ Multi-qubit circuits with entangling gates
3. ✓ Saving and loading datasets
4. ✓ Evaluation datasets with different depths
5. ✓ Randomized benchmarking datasets
6. ✓ Train/validation splits
7. ✓ Dataset statistics and visualization
8. ✓ JSON-based configuration
9. ✓ Clifford and non-Clifford circuits

**Next steps:**
- Use these datasets to train noise-aware quantum circuit models
- Implement reinforcement learning agents for noise characterization
- Compare different noise models and their effects